# Onboarding RAG — LoRA Inference Server (Colab + ngrok)

Serve your fine-tuned **onboarding_lora_v2** on a free Colab T4 GPU. Your local PC runs retrieval; Colab runs generation.

## Before you run
1. **Runtime → Change runtime type → T4 GPU**
2. **Runtime → Restart runtime** before the install cell
3. Upload latest `deploy/colab/` to Drive (`serve.py` + `cuda_setup.py`)
4. LoRA folder on Drive must include `adapter_model.safetensors`
5. Free ngrok token: https://dashboard.ngrok.com/get-started/your-authtoken

## After running all cells
Copy the printed `INFERENCE_URL` into your local `.env`, then:
```bash
python -m inference.rag_engine "your question" --backend remote -v
```

In [ ]:
# --- CONFIG (edit these) ---
LORA_PATH = "/content/drive/MyDrive/onboarding_rag/lora_output/onboarding_lora_v2"
# Fallback if you saved as onboarding_lora during v1 training:
LORA_PATH_FALLBACK = "/content/drive/MyDrive/onboarding_rag/lora_output/onboarding_lora"

# Where serve.py lives on Drive (upload deploy/colab/ folder, or full repo)
DEPLOY_COLAB_DIR = "/content/drive/MyDrive/onboarding_rag/deploy/colab"
PROJECT_ROOT = "/content/drive/MyDrive/onboarding_rag/Onboarding_rag_engine"

# Persist HF downloads on Drive — Colab disk is wiped every new runtime
HF_CACHE = "/content/drive/MyDrive/onboarding_rag/hf_cache"

NGROK_AUTHTOKEN = ""  # paste token from https://dashboard.ngrok.com/get-started/your-authtoken
PORT = 8000
MAX_SEQ_LENGTH = 2048

In [ ]:
# Install Unsloth + CUDA libs — run once, auto-restarts, then run this cell again
from pathlib import Path

MARKER = Path("/content/.onboarding_inference_ready_v3")

if not MARKER.exists():
    pip = lambda *args: get_ipython().run_line_magic("pip", " ".join(args))
    pip("install", "-q", "--upgrade", "pip")
    pip("uninstall", "-y", "torchvision", "torchaudio")
    pip("install", "-q", "-U", "trl>=0.18.2,<=0.24.0,!=0.19.0", "datasets>=3.4.1,<4.4.0")
    pip(
        "install",
        "-q",
        "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git",
    )
    pip("install", "-q", "-U", "bitsandbytes")
    # Colab may need CUDA 12 or 13 nvjitlink — install both (one will match)
    pip("install", "-q", "nvidia-nvjitlink-cu12", "nvidia-nvjitlink-cu13")
    pip("install", "-q", "torchvision")
    MARKER.write_text("ok")
    import os
    print("Packages installed. Restarting runtime (required)...")
    os.kill(os.getpid(), 9)

import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    raise RuntimeError("No GPU — Runtime → Change runtime type → T4 GPU → Restart runtime")

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

lora = Path(LORA_PATH)
if not lora.is_dir():
    lora = Path(LORA_PATH_FALLBACK)
    print(f"Primary LORA_PATH missing, trying fallback: {lora}")

weights = ["adapter_model.safetensors", "adapter_model.bin", "model.safetensors"]
found = [w for w in weights if (lora / w).exists()]
if not found:
    raise FileNotFoundError(
        f"No adapter weights in {lora}. Upload adapter_model.safetensors to Drive."
    )

LORA_PATH = str(lora)
print(f"LoRA OK: {LORA_PATH}")
print(f"Weights: {found}")

In [ ]:
get_ipython().run_line_magic("pip", "install -q fastapi uvicorn[standard] pyngrok nest-asyncio")

# Pre-flight: verify bitsandbytes CUDA before starting server
import sys
from pathlib import Path

for src in (
    Path(DEPLOY_COLAB_DIR) / "cuda_setup.py",
    Path(PROJECT_ROOT) / "deploy/colab/cuda_setup.py",
):
    if src.exists():
        sys.path.insert(0, str(src.parent))
        break

import cuda_setup  # noqa: F401
print("CUDA lib paths:", cuda_setup.CUDA_LIB_PATHS[:5], "...")

import bitsandbytes as bnb  # noqa: F401
print(f"bitsandbytes {bnb.__version__} OK")

import torch
assert torch.cuda.is_available(), "No GPU"
print("CUDA pre-flight passed")

In [ ]:
import os
import shutil
import sys
import threading
import time
from pathlib import Path

import uvicorn

SERVE_DIR = Path("/content/inference_server")
SERVE_DIR.mkdir(exist_ok=True)

candidates = [
    Path(DEPLOY_COLAB_DIR),
    Path(PROJECT_ROOT) / "deploy/colab",
]
copied = False
for deploy_dir in candidates:
    serve_src = deploy_dir / "serve.py"
    cuda_src = deploy_dir / "cuda_setup.py"
    if serve_src.exists() and cuda_src.exists():
        shutil.copy(serve_src, SERVE_DIR / "serve.py")
        shutil.copy(cuda_src, SERVE_DIR / "cuda_setup.py")
        copied = True
        print(f"Copied serve.py + cuda_setup.py from {deploy_dir}")
        break

if not copied:
    raise FileNotFoundError(
        "serve.py / cuda_setup.py not found. Upload deploy/colab/ to Drive."
    )

serve_text = (SERVE_DIR / "serve.py").read_text(encoding="utf-8")
if "drive-cache-4bit-v1" not in serve_text:
    raise RuntimeError(
        "Stale serve.py on Drive — upload the latest deploy/colab/serve.py from your PC, "
        "then re-run this cell. Expected SERVE_REVISION=drive-cache-4bit-v1."
    )
print("serve.py revision check OK (drive-cache-4bit-v1)")

Path(HF_CACHE).mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = HF_CACHE
os.environ["HUGGINGFACE_HUB_CACHE"] = str(Path(HF_CACHE) / "hub")
os.environ["TRANSFORMERS_CACHE"] = str(Path(HF_CACHE) / "transformers")
os.environ["LORA_PATH"] = LORA_PATH
os.environ["PORT"] = str(PORT)
os.environ["MAX_SEQ_LENGTH"] = str(MAX_SEQ_LENGTH)
print(f"HF cache → {HF_CACHE} (survives Colab restarts)")

sys.path.insert(0, str(SERVE_DIR))
import serve  # noqa: E402


def _run_server():
    uvicorn.run(serve.app, host="0.0.0.0", port=PORT, log_level="info")


server_thread = threading.Thread(target=_run_server, daemon=True)
server_thread.start()
print(
    "Starting server. First run: ~5GB 4bit base → Drive (once). Later runs: load from Drive."
)

import urllib.request

for i in range(600):
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{PORT}/health", timeout=5) as r:
            data = r.read().decode()
            if '"model_loaded":true' in data.replace(" ", ""):
                print("Model loaded:", data)
                break
            if i % 15 == 0:
                print(f"  waiting for model... ({i}s)")
    except Exception:
        if i % 15 == 0:
            print(f"  waiting for server... ({i}s)")
    time.sleep(1)
else:
    print("Still loading — wait for model_loaded:true in /health, then run ngrok cell")


In [ ]:
from pyngrok import ngrok, conf

if not NGROK_AUTHTOKEN:
    raise ValueError("Set NGROK_AUTHTOKEN in the config cell")

conf.get_default().auth_token = NGROK_AUTHTOKEN

# Close any existing tunnels from a previous run
for t in ngrok.get_tunnels():
    ngrok.disconnect(t.public_url)

public_url = ngrok.connect(PORT, bind_tls=True).public_url
INFERENCE_URL = public_url.rstrip("/")

print("=" * 60)
print("INFERENCE_URL =", INFERENCE_URL)
print("=" * 60)
print("\nAdd to your local .env:")
print(f"INFERENCE_URL={INFERENCE_URL}")
print("\nThen on your PC:")
print('python -m inference.rag_engine "what arguments does HTTPException take?" --backend remote -v')

In [ ]:
# Quick test through ngrok
import json
import urllib.request

payload = json.dumps({
    "messages": [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Say hello in one sentence."},
    ],
    "max_tokens": 64,
    "temperature": 0.2,
}).encode()

req = urllib.request.Request(
    f"{INFERENCE_URL}/generate",
    data=payload,
    headers={"Content-Type": "application/json"},
    method="POST",
)
with urllib.request.urlopen(req, timeout=120) as resp:
    print(json.loads(resp.read().decode()))

## Keep this notebook open

- Closing Colab stops the server
- If ngrok URL changes, update `INFERENCE_URL` in your local `.env`
- Re-run the **ngrok** cell if the tunnel drops